# Lab Tasks

In this task, we will look at creating networks from large-scale real-world data.

The dataset used for this task consists of flight records retrieved from the [US Bureau of Transportation Statistics website](https://www.transtats.bts.gov). The data covers the period Q1 and Q2 2016, and includes each flight's origin, destination, along with other relevant metadata. The raw data is provided as a single CSV file (airstats-2016.csv).

### Task 1

Load the flight record data from the file airstats-2016.csv into a Pandas DataFrame, and apply the following filtering steps to the DataFrame:

1. Only include records where the flight origin and destination were both in the United States.
2. Only include records from the time periods Q1 2016 and Q2 2016.
3. Only include records where the reported distance between the origin and destination was at least 20 miles.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('airstats-2016.csv')
df.head()

,DISTANCE,UNIQUE_CARRIER,UNIQUE_CARRIER_NAME,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_COUNTRY,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_COUNTRY,YEAR,QUARTER,MONTH
0,226,UA,United Air Lines Inc.,12889,LAS,"Las Vegas, NV",US,14908,SNA,"Santa Ana, CA",US,2016,3,9
1,1943,UA,United Air Lines Inc.,12889,LAS,"Las Vegas, NV",US,16271,YYZ,"Toronto, Canada",CA,2016,3,9
2,1814,UA,United Air Lines Inc.,12892,LAX,"Los Angeles, CA",US,12339,IND,"Indianapolis, IN",US,2016,3,9
3,1363,UA,United Air Lines Inc.,12892,LAX,"Los Angeles, CA",US,13198,MCI,"Kansas City, MO",US,2016,3,9
4,1670,UA,United Air Lines Inc.,12892,LAX,"Los Angeles, CA",US,13495,MSY,"New Orleans, LA",US,2016,3,9


In [3]:
df_filtered = df.loc[
    (df['DEST_COUNTRY'] == 'US') & 
    (df['ORIGIN_COUNTRY'] == 'US') &
    (df['QUARTER'] < 3) &
    (df['DISTANCE'] >= 20)
]

In [4]:
df_filtered.head()

,DISTANCE,UNIQUE_CARRIER,UNIQUE_CARRIER_NAME,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_COUNTRY,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_COUNTRY,YEAR,QUARTER,MONTH
4679,374,09Q,"Swift Air, LLC",10185,AEX,"Alexandria, LA",US,14683,SAT,"San Antonio, TX",US,2016,1,1
4680,591,09Q,"Swift Air, LLC",10397,ATL,"Atlanta, GA",US,13232,MDW,"Chicago, IL",US,2016,1,1
4681,276,09Q,"Swift Air, LLC",10466,AZA,"Phoenix, AZ",US,12889,LAS,"Las Vegas, NV",US,2016,1,1
4682,1950,09Q,"Swift Air, LLC",10466,AZA,"Phoenix, AZ",US,13303,MIA,"Miami, FL",US,2016,1,1
4683,294,09Q,"Swift Air, LLC",10469,AZO,"Kalamazoo, MI",US,12945,LEX,"Lexington, KY",US,2016,1,1


### Task 2

Create an **unweighted directed** network from the Pandas DataFrame.

Use the three-letter IATA airport codes for the origin and destination as the node identifiers. Also add the airport's city name as an attribute for each node.

In [5]:
import networkx as nx
import itertools

In [27]:
G = nx.from_pandas_edgelist(
    df_filtered,
    source='ORIGIN',
    target='DEST',
    create_using=nx.DiGraph()
)

for _, row in df_filtered.iterrows():
    G.nodes[row['ORIGIN']]['city'] = row['ORIGIN_CITY_NAME']
    G.nodes[row['DEST']]['city'] = row['DEST_CITY_NAME']

G.nodes['LAX']['city']

'Los Angeles, CA'

### Task 3

Characterise the unweighted directed network from Task 2, examining:
      
1. How many nodes and edges are in the network?
2. The connectedness of the network (i.e., density and number of components).
3. Identify frequent origin and destination airports in the network (i.e., in-degree and out-degree).
4. Identify key hub airports in the network (i.e., betweenness centrality).

In [7]:
print(f'Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges')

Graph has 1043 nodes and 17644 edges


In [8]:
density = nx.density(G)

density

0.01623472818515908

In [9]:
num_components = nx.number_weakly_connected_components(G)
num_components

3

In [10]:
out_deg = G.out_degree()
sorted(out_deg, key=lambda x: x[1], reverse=True)[:10]


[('ORD', 205),
 ('ATL', 193),
 ('DEN', 187),
 ('ANC', 183),
 ('DFW', 175),
 ('MSP', 171),
 ('LAS', 162),
 ('MEM', 162),
 ('DTW', 152),
 ('LAX', 151)]

In [11]:
in_deg = G.in_degree()
sorted(in_deg, key=lambda x: x[1], reverse=True)[:10]


[('ORD', 200),
 ('ATL', 187),
 ('DEN', 184),
 ('DFW', 176),
 ('MSP', 175),
 ('LAS', 155),
 ('MEM', 152),
 ('SDF', 151),
 ('IAH', 151),
 ('LAX', 147)]

In [12]:
betweenness = nx.betweenness_centrality(G)
sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]


[('ANC', 0.28162344313544047),
 ('FAI', 0.07801074284298082),
 ('SEA', 0.06823583856800133),
 ('HPN', 0.06463146673434744),
 ('ORD', 0.05834560698534908),
 ('DEN', 0.052815742670521026),
 ('MSP', 0.0461657128046315),
 ('ADQ', 0.035831177742632336),
 ('DFW', 0.03375168841456716),
 ('ATL', 0.03308372915448028)]

In [13]:
top_hubs = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]

[(node, G.nodes[node]['city'], score) for node, score in top_hubs]


[('ANC', 'Anchorage, AK', 0.28162344313544047),
 ('FAI', 'Fairbanks, AK', 0.07801074284298082),
 ('SEA', 'Seattle, WA', 0.06823583856800133),
 ('HPN', 'White Plains, NY', 0.06463146673434744),
 ('ORD', 'Chicago, IL', 0.05834560698534908)]

### Task 4

Now create an alternative **weighted directed** network from the pandas DataFrame.

In [28]:
G_weighted = nx.DiGraph()

for _, row in df_filtered.iterrows():
    origin = row['ORIGIN']
    dest = row['DEST']

    # add nodes with city attributes
    if not G_weighted.has_node(origin):
        G_weighted.add_node(origin, city=row['ORIGIN_CITY_NAME'])
    if not G_weighted.has_node(dest):
        G_weighted.add_node(dest, city=row['DEST_CITY_NAME'])

    # increment edge weight
    if G_weighted.has_edge(origin, dest):
        G_weighted[origin][dest]['weight'] += 1
    else:
        G_weighted.add_edge(origin, dest, weight=1)


### Task 5

Based on the weighted directed network, identify:
    
1. The most frequent routes in the network.
2. The most frequent origin and destination airports in the network, considering edge weights.

In [29]:
top_routes = sorted(
    G_weighted.edges(data=True),
    key=lambda x: x[2]['weight'],
    reverse=True
)[:10]

top_routes


[('MSP', 'ORD', {'weight': 78}),
 ('ORD', 'MSP', {'weight': 75}),
 ('DEN', 'SLC', {'weight': 71}),
 ('DTW', 'ORD', {'weight': 70}),
 ('ORD', 'DTW', {'weight': 69}),
 ('SLC', 'DEN', {'weight': 63}),
 ('CVG', 'ORD', {'weight': 61}),
 ('ORD', 'MCI', {'weight': 60}),
 ('ORD', 'ATL', {'weight': 60}),
 ('LAX', 'SFO', {'weight': 60})]